# Barter-RS: Instruments & Index Management

This notebook demonstrates the `barter-instrument` crate — the foundation for
defining exchanges, assets, and instruments with O(1) indexed lookups.

## Topics Covered
- Defining exchanges, assets, and instruments
- Building `IndexedInstruments` for compact indexed access
- Using `ExchangeIndex`, `AssetIndex`, `InstrumentIndex` for O(1) lookups
- Instrument kinds: Spot, Perpetual, Future, Option
- The `Keyed<Key, Value>` utility type

In [ ]:
:dep barter-instrument = { version = "0.3" }
:dep serde_json = "1.0"

## 1. Core Types

Barter uses strongly-typed wrappers for all financial primitives:

- **`ExchangeId`** — identifies an exchange (e.g., `BinanceSpot`, `Okx`)
- **`Asset`** — a tradeable asset (e.g., `btc`, `usdt`)
- **`Instrument`** — a specific trading pair on an exchange
- **`Underlying`** — the base/quote pair (e.g., BTC/USDT)
- **`InstrumentKind`** — Spot, Perpetual, Future, or Option

In [ ]:
use barter_instrument::{
    Underlying,
    exchange::ExchangeId,
    instrument::{Instrument, InstrumentIndex, kind::InstrumentKind},
    asset::AssetIndex,
    index::IndexedInstruments,
};

// Define instruments across multiple exchanges and kinds
let instruments = IndexedInstruments::builder()
    // Binance Spot BTC/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_btc_usdt",
        "BTCUSDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    // Binance Spot ETH/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::BinanceSpot,
        "binance_spot_eth_usdt",
        "ETHUSDT",
        Underlying::new("eth", "usdt"),
        None,
    ))
    // OKX Spot BTC/USDT
    .add_instrument(Instrument::spot(
        ExchangeId::Okx,
        "okx_spot_btc_usdt",
        "BTC-USDT",
        Underlying::new("btc", "usdt"),
        None,
    ))
    .build();

println!("Built IndexedInstruments:");
println!("  Exchanges: {}", instruments.exchanges().len());
println!("  Assets:    {}", instruments.assets().len());
println!("  Instruments: {}", instruments.instruments().len());

## 2. Indexed Lookups

All entities are assigned compact integer indices at build time. This enables
O(1) array-based lookups instead of HashMap access — critical for hot-path
trading logic.

```
ExchangeId::BinanceSpot  →  ExchangeIndex(0)
ExchangeId::Okx          →  ExchangeIndex(1)
"btc"                    →  AssetIndex(0)
"usdt"                   →  AssetIndex(1)
"BTCUSDT" on Binance     →  InstrumentIndex(0)
```

In [ ]:
// Access instruments by index
println!("Instruments by index:");
for (idx, instrument) in instruments.instruments().iter().enumerate() {
    println!(
        "  InstrumentIndex({idx}): exchange={}, name={}, kind={:?}, base={}, quote={}",
        instrument.exchange,
        instrument.name_exchange,
        instrument.kind,
        instrument.underlying.base,
        instrument.underlying.quote,
    );
}

println!("\nAssets by index:");
for (idx, asset) in instruments.assets().iter().enumerate() {
    println!("  AssetIndex({idx}): {asset:?}");
}

println!("\nExchanges by index:");
for (idx, exchange) in instruments.exchanges().iter().enumerate() {
    println!("  ExchangeIndex({idx}): {exchange:?}");
}

## 3. Instrument Kinds

Barter supports multiple instrument kinds for different market types:

In [ ]:
use barter_instrument::instrument::kind::InstrumentKind;

// Available instrument kinds
let kinds = [
    InstrumentKind::Spot,
    InstrumentKind::Perpetual,
];

println!("Supported InstrumentKinds:");
for kind in &kinds {
    println!("  - {kind:?}");
}

println!("\nAdditionally: Future (with expiry) and Option (with strike, expiry, option_kind)");

## 4. JSON Serialisation

Instruments serialise cleanly to JSON, which is how `SystemConfig` stores them.

In [ ]:
let instrument = Instrument::spot(
    ExchangeId::BinanceSpot,
    "binance_spot_btc_usdt",
    "BTCUSDT",
    Underlying::new("btc", "usdt"),
    None,
);

let json = serde_json::to_string_pretty(&instrument)
    .expect("failed to serialise");

println!("Instrument as JSON:\n{json}");

// Round-trip deserialisation
let roundtrip: Instrument = serde_json::from_str(&json)
    .expect("failed to deserialise");

println!("\nRound-trip successful: {} on {}", roundtrip.name_exchange, roundtrip.exchange);

## 5. The `Keyed<Key, Value>` Pattern

Barter uses `Keyed<K, V>` throughout to associate values with their index keys.
This is a simple but powerful pattern for maintaining type-safe key-value
associations without using maps.

```rust
pub struct Keyed<Key, Value> {
    pub key: Key,
    pub value: Value,
}
```

This is used for:
- `Keyed<InstrumentIndex, Position>` — a position tagged with its instrument
- `Keyed<AssetIndex, Balance>` — a balance tagged with its asset
- `Keyed<ExchangeIndex, Vec<Order>>` — orders grouped by exchange

In [ ]:
use barter_instrument::Keyed;
use barter_instrument::instrument::InstrumentIndex;

// Example: tagging a value with its instrument index
let position_pnl = Keyed {
    key: InstrumentIndex(0),
    value: 1500.0_f64, // PnL in quote currency
};

println!(
    "Instrument {:?} has PnL: {:.2}",
    position_pnl.key, position_pnl.value
);

## Architecture: Why Indexed?

Traditional trading systems use `HashMap<String, T>` for lookups. Barter replaces
this with indexed arrays:

| Approach | Lookup Cost | Cache Behaviour | Memory |
|----------|-------------|-----------------|--------|
| `HashMap<String, T>` | O(1) amortised, hash + compare | Cache-unfriendly (pointer chasing) | Higher (string allocs) |
| `Vec<T>` + `Index(usize)` | O(1) guaranteed | Cache-friendly (contiguous) | Minimal |

The `IndexedInstruments` builder resolves all string identifiers to compact
indices at startup. All hot-path code then uses `ExchangeIndex`, `AssetIndex`,
and `InstrumentIndex` for direct array access.